# Optimización de algoritmos con Boosting para predicción de diabetes

En este proyecto se trabaja sobre un problema de clasificación relacionado con la predicción de diabetes, retomando el dataset procesado en proyectos anteriores del recorrido. La idea central es evaluar si un modelo de boosting puede mejorar el desempeño obtenido previamente con un árbol de decisión y con random forest.

El trabajo sigue una lógica progresiva: partir de un conjunto de datos ya preparado, entrenar un modelo base de boosting, ajustar hiperparámetros, analizar el impacto de esos cambios y comparar los resultados finales con los modelos anteriores para decidir cuál conviene conservar.

## Contexto del problema

En los proyectos anteriores se construyeron modelos basados en árbol de decisión y random forest para abordar el mismo problema de clasificación. Ese recorrido permitió entender el comportamiento del dataset, trabajar su versión procesada y obtener una primera referencia del rendimiento alcanzable.

En este punto, el objetivo ya no es solo entrenar otro modelo, sino comprobar si una estrategia de ensamblado secuencial como boosting puede capturar mejor los patrones del conjunto de datos y corregir parte de los errores que seguían presentes en los modelos anteriores.

## Objetivo general

El objetivo de este notebook es construir y analizar un modelo de boosting para predicción de diabetes, evaluando si aporta una mejora real frente a los modelos trabajados previamente.

De forma más concreta, se busca:

- entrenar un modelo base de boosting sobre el dataset procesado;
- ajustar hiperparámetros relevantes para estudiar su impacto en el rendimiento;
- evaluar el modelo con métricas de clasificación;
- comparar sus resultados con los obtenidos con árbol de decisión y random forest;
- justificar, con base en evidencia, qué modelo resulta más conveniente para este problema.

## 1. Librerías y preparación del entorno

### Reutilización del dataset procesado

Para este proyecto se utilizan los mismos conjuntos `X_train`, `X_test`, `y_train` y `y_test` guardados en `data/processed/` en el proyecto anterior. Trabajar con los mismos datos permite centrar el análisis en el comportamiento del boosting y comparar su rendimiento con árbol de decisión y random forest en condiciones equivalentes.

In [1]:
import warnings
from pathlib import Path
import os

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_STATE = 42

In [2]:
# Raíz del proyecto
current_path = Path.cwd().resolve()
BASE_DIR = current_path.parent if current_path.name == "src" else current_path

# Rutas de trabajo
DATA_DIR = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"

print("Directorio base:", BASE_DIR)
print("Ruta de datos procesados:", DATA_DIR)
print("Ruta de modelos:", MODELS_DIR)

Directorio base: /workspaces/gondelles-boosting-algorithm-project
Ruta de datos procesados: /workspaces/gondelles-boosting-algorithm-project/data/processed
Ruta de modelos: /workspaces/gondelles-boosting-algorithm-project/models


## Breve marco conceptual: boosting

Boosting es una técnica de ensamblado en la que varios modelos débiles se entrenan de forma secuencial. A diferencia de enfoques donde los modelos se construyen de manera independiente, aquí cada nuevo modelo intenta corregir parte de los errores cometidos por el anterior.

En problemas de clasificación, este enfoque puede ser útil cuando un único árbol no alcanza suficiente capacidad de generalización, pero todavía hay estructura en los datos que puede aprovecharse mejor. En ese sentido, boosting suele ofrecer modelos más precisos que un árbol individual, aunque también exige más cuidado con el ajuste de hiperparámetros para evitar sobreajuste.

En este proyecto se utiliza `XGBoost`, una implementación eficiente de boosting muy extendida en problemas tabulares.

## ¿Por qué probar boosting en este caso?

Tiene sentido probar boosting en este problema por dos motivos. El primero es metodológico: forma parte natural del recorrido después de haber trabajado con árbol de decisión y random forest. El segundo es técnico: si los modelos anteriores capturaron parte del patrón, pero todavía dejaron margen de mejora, un ensamblado secuencial puede ayudar a corregir errores que un modelo individual o un bosque más plano no terminan de resolver.

Eso no implica asumir de antemano que boosting será mejor. Justamente por eso este notebook no se limita a entrenar el modelo, sino que compara su comportamiento con los modelos previos y analiza si la mejora, en caso de existir, compensa el aumento de complejidad y el riesgo de sobreajuste.

## Carga del dataset procesado

En este proyecto no se parte del dataset en bruto. Se reutiliza el conjunto de datos ya trabajado en la etapa anterior, donde se realizó el EDA, la limpieza necesaria y la división entre entrenamiento y prueba.

Esto permite concentrar el análisis en la fase de modelado y comparación, manteniendo consistencia con los proyectos previos y evitando repetir pasos ya resueltos en el flujo del trabajo.

## Modelado inicial

Como punto de partida, se entrena un modelo base de `XGBClassifier`. La intención de esta primera versión no es encontrar ya el mejor resultado posible, sino establecer una referencia inicial sobre la que luego tenga sentido optimizar.

Trabajar con un baseline ayuda a responder una pregunta simple pero importante: antes de ajustar hiperparámetros, ¿qué rendimiento ofrece el modelo con una configuración razonable de partida? A partir de esa referencia, cualquier mejora posterior puede interpretarse con más claridad.

## Optimización de hiperparámetros

Después del modelo base, se ajustan algunos hiperparámetros del boosting para analizar cómo cambian los resultados. En este tipo de modelos, parámetros como `n_estimators`, `learning_rate`, `max_depth`, `subsample` o `colsample_bytree` pueden modificar bastante el equilibrio entre capacidad de aprendizaje, estabilidad y riesgo de sobreajuste.

La idea no es probar combinaciones de forma arbitraria, sino observar qué cambios aportan mejora real y cuáles solo añaden complejidad sin beneficio claro. Por eso, además de identificar la mejor configuración, interesa analizar la tendencia general de los resultados.

## Criterio de evaluación

Para evaluar el modelo no alcanza con mirar una sola métrica de forma aislada. El `accuracy` permite tener una visión general del porcentaje de aciertos, pero no siempre refleja bien qué ocurre en cada clase.

Por eso también se utiliza el `classification report`, que permite revisar `precision`, `recall` y `f1-score`, y la matriz de confusión, que ayuda a detectar en qué tipo de casos se concentra el error del modelo.

Este enfoque es especialmente útil cuando el objetivo no es solo mejorar un número global, sino entender cómo está tomando decisiones el modelo.

## Comparación con modelos anteriores

Una vez evaluado el modelo de boosting, los resultados se comparan con los obtenidos previamente con árbol de decisión y random forest. Esta comparación es necesaria para que la elección final no dependa únicamente de una intuición sobre el algoritmo, sino de evidencia concreta.

El criterio de comparación no se limita al `accuracy`. También importa observar si la mejora es consistente, si el modelo mantiene un comportamiento razonable por clase y si el aumento de complejidad queda justificado por el resultado final.

## Fortalezas y debilidades del modelo de boosting

Una de las principales fortalezas del boosting es su capacidad para mejorar progresivamente sobre errores anteriores, lo que suele traducirse en un buen rendimiento en datos tabulares. Además, permite capturar relaciones más complejas que un árbol individual.

Como contrapartida, es un modelo más sensible al ajuste de hiperparámetros y puede sobreajustarse si se incrementa demasiado la complejidad. También es menos directo de interpretar que un árbol de decisión simple.

Por eso, la decisión final no debería basarse solo en si obtiene el mejor resultado numérico, sino en si esa mejora es suficientemente clara y estable como para justificar su uso frente a alternativas más simples.

## Conclusión final

A partir de los resultados obtenidos, el modelo seleccionado fue **[COMPLETAR CON MODELO ELEGIDO]**.

La decisión se apoya en los siguientes puntos:

- `accuracy`: [COMPLETAR CON RESULTADO REAL]
- comportamiento por clase: [COMPLETAR CON HALLAZGO PRINCIPAL]
- comparación frente a árbol de decisión y random forest: [COMPLETAR CON COMPARACIÓN REAL]
- balance entre rendimiento y complejidad: [COMPLETAR CON JUSTIFICACIÓN]

En conjunto, los resultados sugieren que **[COMPLETAR CON INTERPRETACIÓN FINAL]**. Aun así, la elección no debe entenderse como definitiva para cualquier contexto, sino como la mejor opción dentro del alcance de este proyecto y con el dataset trabajado.